# Lab 11: Data Wrangling - Complete Pipeline
## IBM Data Analytics Capstone Project
### Objective: Clean, transform, encode, normalize and engineer features from the Stack Overflow Developer Survey dataset

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the dataset
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

print(" Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

 Dataset loaded successfully!
Shape: (65437, 114)


,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,SurveyLength,SurveyEase,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,Under 18 years old,"Employed, full-time",Remote,Apples,Hobby,Primary/elementary school,Books / Physical media,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,NaN,NaN
3,4,I am learning to code,18-24 years old,"Student, full-time",NaN,Apples,NaN,Some college/university study without earning ...,"Other online resources (e.g., videos, blogs, f...",Stack Overflow;How-to videos;Interactive tutorial,...,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Easy,NaN,NaN
4,5,I am a developer by profession,18-24 years old,"Student, full-time",NaN,Apples,NaN,"Secondary school (e.g. American high school, G...","Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Written Tutorial...,...,NaN,NaN,NaN,NaN,NaN,NaN,Too short,Easy,NaN,NaN


In [2]:
# 2.1 Column data types, counts and missing values
print("Dataset Info:")
print(df.dtypes.value_counts())
print("\nTotal missing values per column (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))

# 2.2 Basic statistics for numerical columns
print("\nNumerical columns summary:")
df.describe()

Dataset Info:
object     100
float64     13
int64        1
Name: count, dtype: int64

Total missing values per column (top 10):
AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
ConvertedCompYearly              42002
AIToolNot interested in Using    41023
AINextMore integrated            41009
dtype: int64

Numerical columns summary:


,ResponseId,CompTotal,WorkExp,JobSatPoints_1,JobSatPoints_4,JobSatPoints_5,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,ConvertedCompYearly,JobSat
count,65437.000000,3.374000e+04,29658.000000,29324.000000,29393.000000,29411.000000,29450.000000,29448.00000,29456.000000,29456.000000,29450.000000,29445.000000,2.343500e+04,29126.000000
mean,32719.000000,2.963841e+145,11.466957,18.581094,7.522140,10.060857,24.343232,22.96522,20.278165,16.169432,10.955713,9.953948,8.615529e+04,6.935041
std,18890.179119,5.444117e+147,9.168709,25.966221,18.422661,21.833836,27.089360,27.01774,26.108110,24.845032,22.906263,21.775652,1.867570e+05,2.088259
min,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.000000e+00,0.000000
25%,16360.000000,6.000000e+04,4.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,3.271200e+04,6.000000
50%,32719.000000,1.100000e+05,9.000000,10.000000,0.000000,0.000000,20.000000,15.00000,10.000000,5.000000,0.000000,0.000000,6.500000e+04,7.000000
75%,49078.000000,2.500000e+05,16.000000,22.000000,5.000000,10.000000,30.000000,30.00000,25.000000,20.000000,10.000000,10.000000,1.079715e+05,8.000000
max,65437.000000,1.000000e+150,50.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,1.625660e+07,10.000000


In [3]:
# Identify and remove duplicates
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)
print("Duplicates handled!")

Duplicate rows: 0
Shape after removing duplicates: (65437, 114)
Duplicates handled!


In [4]:
# 3.1 Identify inconsistent entries in Country
print("Total unique countries:", df['Country'].nunique())
print("\nTop 10 countries:")
print(df['Country'].value_counts().head(10))

# 3.2 Standardize EdLevel by shortening labels
edlevel_mapping = {
    "Bachelor's degree (B.A., B.S., B.Eng., etc.)": "Bachelor's",
    "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": "Master's",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional/PhD",
    "Some college/university study without earning a degree": "Some College",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary School",
    "Associate degree (A.A., A.S., etc.)": "Associate",
    "Primary/elementary school": "Primary School",
    "Something else": "Other"
}
df['EdLevel'] = df['EdLevel'].map(edlevel_mapping).fillna(df['EdLevel'])
print("\nStandardized EdLevel values:")
print(df['EdLevel'].value_counts())

Total unique countries: 185

Top 10 countries:
Country
United States of America                                11095
Germany                                                  4947
India                                                    4231
United Kingdom of Great Britain and Northern Ireland     3224
Ukraine                                                  2672
France                                                   2110
Canada                                                   2104
Poland                                                   1534
Netherlands                                              1449
Brazil                                                   1375
Name: count, dtype: int64

Standardized EdLevel values:
EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)       24942
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)    15557
Some College                                        7651
Secondary School                                    5793
Professional/PhD                 

In [5]:
# 4.1 One-hot encode Employment column
employment_dummies = pd.get_dummies(df['Employment'], prefix='Emp')
df = pd.concat([df, employment_dummies], axis=1)

print("One-hot encoding applied to Employment!")
print("New columns added:", list(employment_dummies.columns[:5]), "...")
print("New shape:", df.shape)

One-hot encoding applied to Employment!
New columns added: ['Emp_Employed, full-time', 'Emp_Employed, full-time;Employed, part-time', 'Emp_Employed, full-time;Independent contractor, freelancer, or self-employed', 'Emp_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time', 'Emp_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time;Retired'] ...
New shape: (65437, 224)


In [6]:
# 5.1 Columns with highest missing values
print("Top 10 columns with missing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

# 5.2 Impute ConvertedCompYearly with median
median_comp = df['ConvertedCompYearly'].median()
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(median_comp)
print("\nConvertedCompYearly imputed with median:", median_comp)
print("Missing after imputation:", df['ConvertedCompYearly'].isnull().sum())

# 5.3 Impute RemoteWork with most frequent value
most_frequent_remote = df['RemoteWork'].value_counts().idxmax()
df['RemoteWork'] = df['RemoteWork'].fillna(most_frequent_remote)
print("\nRemoteWork imputed with:", most_frequent_remote)
print("Missing after imputation:", df['RemoteWork'].isnull().sum())

Top 10 columns with missing values:
AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
ConvertedCompYearly              42002
AIToolNot interested in Using    41023
AINextMore integrated            41009
dtype: int64

ConvertedCompYearly imputed with median: 65000.0
Missing after imputation: 0

RemoteWork imputed with: Hybrid (some remote, some in-person)
Missing after imputation: 0


In [7]:
# 6.1 Min-Max Scaling
min_val = df['ConvertedCompYearly'].min()
max_val = df['ConvertedCompYearly'].max()
df['ConvertedCompYearly_MinMax'] = (df['ConvertedCompYearly'] - min_val) / (max_val - min_val)
print("Min-Max Scaling applied!")

# 6.2 Log Transform
df['ConvertedCompYearly_Log'] = np.log1p(df['ConvertedCompYearly'])
print("Log Transform applied!")
print("\nComparison:")
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax', 
          'ConvertedCompYearly_Log']].describe())

Min-Max Scaling applied!
Log Transform applied!

Comparison:
       ConvertedCompYearly  ConvertedCompYearly_MinMax  \
count         6.543700e+04                65437.000000   
mean          7.257636e+04                    0.004464   
std           1.122207e+05                    0.006903   
min           1.000000e+00                    0.000000   
25%           6.500000e+04                    0.003998   
50%           6.500000e+04                    0.003998   
75%           6.500000e+04                    0.003998   
max           1.625660e+07                    1.000000   

       ConvertedCompYearly_Log  
count             65437.000000  
mean                 10.976053  
std                   0.851456  
min                   0.693147  
25%                  11.082158  
50%                  11.082158  
75%                  11.082158  
max                  16.604010  


In [8]:
# 7.1 Create ExperienceLevel based on YearsCodePro
print("YearsCodePro unique values sample:")
print(df['YearsCodePro'].value_counts().head(10))

# Convert to numeric, handle non-numeric values
df['YearsCodePro_Clean'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')

# Create ExperienceLevel categories
def categorize_experience(years):
    if pd.isna(years):
        return 'Unknown'
    elif years <= 2:
        return 'Junior'
    elif years <= 5:
        return 'Mid-level'
    elif years <= 10:
        return 'Senior'
    else:
        return 'Expert'

df['ExperienceLevel'] = df['YearsCodePro_Clean'].apply(categorize_experience)

print("\nExperienceLevel distribution:")
print(df['ExperienceLevel'].value_counts())

YearsCodePro unique values sample:
YearsCodePro
2                   4168
3                   4093
5                   3526
10                  3251
4                   3215
Less than 1 year    2856
6                   2843
1                   2639
8                   2549
7                   2517
Name: count, dtype: int64

ExperienceLevel distribution:
ExperienceLevel
Expert       18410
Unknown      16733
Senior       12653
Mid-level    10834
Junior        6807
Name: count, dtype: int64
